In [21]:
import os
import json
import pandas as pd

from dotenv import load_dotenv
from google.oauth2 import service_account
from google.cloud import storage

In [22]:
# Load environment variables from .env file
load_dotenv()

# Access the variables using os.getenv()
service_account_key = os.getenv("GCP_SERVICE_ACCOUNT_KEY")
project_id = os.getenv("GCP_PROJECT_ID")
bucket_name = os.getenv("GCP_BUCKET_NAME")

In [23]:
file_name = "../scraping/data/sfpd_compstat.json"

In [24]:
with open(file_name, "r") as f:
    content = json.load(f)

In [25]:
df = pd.DataFrame(content)
df.head()

,report_period,report_year,report_month,district,section,category,same_month_prior_year,current_month,previous_month,ytd_prior_year,ytd_current,source_url,scraped_at
0,2023-01,2023,1,CITYWIDE,part1_violent,homicide,3,5,4,3,5,https://www.sanfranciscopolice.org/sites/defau...,2026-10-11T19:46:08+00:00
1,2023-01,2023,1,CITYWIDE,part1_violent,rape,19,10,20,19,10,https://www.sanfranciscopolice.org/sites/defau...,2026-10-11T19:46:08+00:00
2,2023-01,2023,1,CITYWIDE,part1_violent,robbery,178,197,160,178,197,https://www.sanfranciscopolice.org/sites/defau...,2026-10-11T19:46:08+00:00
3,2023-01,2023,1,CITYWIDE,part1_violent,aggravated_assault,188,191,197,188,191,https://www.sanfranciscopolice.org/sites/defau...,2026-10-11T19:46:08+00:00
4,2023-01,2023,1,CITYWIDE,part1_violent,human_trafficking_sex_act,2,1,1,2,1,https://www.sanfranciscopolice.org/sites/defau...,2026-10-11T19:46:08+00:00


In [26]:
df = df[df["district"] != "CITYWIDE"].reset_index(drop=True)
df["district"].value_counts()

district
CENTRAL       1075
PARK          1070
RICHMOND      1066
MISSION       1063
BAYVIEW       1062
INGLESIDE     1061
NORTHERN      1060
TARAVAL       1058
SOUTHERN      1057
TENDERLOIN    1053
Name: count, dtype: int64

In [27]:
crime_month_summary = (
    df.groupby("report_period")
      .agg(
          rows=("report_period", "size"),
          districts=("district", "nunique"),
          categories=("category", "nunique")
      )
)

crime_analysis = df[
    [
        "report_period",
        "district",
        "section",
        "category",
        "current_month"
    ]
].copy()

crime_analysis.head()

,report_period,district,section,category,current_month
0,2023-01,CENTRAL,part1_violent,homicide,0
1,2023-01,CENTRAL,part1_violent,rape,0
2,2023-01,CENTRAL,part1_violent,robbery,20
3,2023-01,CENTRAL,part1_property,larceny_theft,673
4,2023-01,CENTRAL,part1_property,theft_from_vehicle,504


In [28]:
overall_crime = crime_analysis[
    crime_analysis["category"] == "total_part1"
].copy()

overall_crime.head()

,report_period,district,section,category,current_month
8,2023-01,CENTRAL,part1_total,total_part1,804
18,2023-01,SOUTHERN,part1_total,total_part1,287
32,2023-01,BAYVIEW,part1_total,total_part1,290
47,2023-01,MISSION,part1_total,total_part1,424
61,2023-01,NORTHERN,part1_total,total_part1,644


In [29]:
print(overall_crime.shape)

print("\nRows per month:")
print(overall_crime["report_period"].value_counts().sort_index())

print("\nDistricts:")
print(sorted(overall_crime["district"].unique()))

(438, 5)

Rows per month:
report_period
2023-01    10
2023-02    10
2023-03    10
2023-04    10
2023-05    10
2023-06    10
2023-07    10
2023-08    10
2023-09    10
2023-10    10
2023-11    10
2023-12    10
2024-01    10
2024-02    10
2024-03    10
2024-04     9
2024-05    10
2024-06    10
2024-07    10
2024-08    10
2024-09    10
2024-10    10
2024-11     9
2024-12    10
2025-01    10
2025-02    10
2025-03    10
2025-04    10
2025-05    10
2025-06    10
2025-07    10
2025-08    10
2025-09    10
2025-10    10
2025-11    10
2025-12    10
2026-01    10
2026-02    10
2026-03    10
2026-04    10
2026-05    10
2026-06    10
2026-07    10
2026-08    10
Name: count, dtype: int64

Districts:
['BAYVIEW', 'CENTRAL', 'INGLESIDE', 'MISSION', 'NORTHERN', 'PARK', 'RICHMOND', 'SOUTHERN', 'TARAVAL', 'TENDERLOIN']


In [30]:
for period in ["2024-04", "2024-11"]:
    month_df = crime_analysis[crime_analysis["report_period"] == period]

    all_districts = set(month_df["district"])
    total_districts = set(
        month_df.loc[
            month_df["category"] == "total_part1",
            "district"
        ]
    )

    print(period, "missing total_part1 for:", sorted(all_districts - total_districts))

2024-04 missing total_part1 for: ['CENTRAL']
2024-11 missing total_part1 for: ['BAYVIEW']


In [31]:
check_missing_totals = crime_analysis[
    (
        (crime_analysis["report_period"] == "2024-04") &
        (crime_analysis["district"] == "CENTRAL")
    )
    |
    (
        (crime_analysis["report_period"] == "2024-11") &
        (crime_analysis["district"] == "BAYVIEW")
    )
]

check_missing_totals[
    check_missing_totals["category"].isin([
        "total_part1_violent",
        "total_part1_property",
        "total_part1"
    ])
]

,report_period,district,section,category,current_month
2978,2024-04,CENTRAL,part1_violent,total_part1_violent,36
2985,2024-04,CENTRAL,part1_property,total_part1_property,295
4846,2024-11,BAYVIEW,part1_violent,total_part1_violent,42
4853,2024-11,BAYVIEW,part1_property,total_part1_property,169


In [32]:
missing_totals = (
    check_missing_totals[
        check_missing_totals["category"].isin([
            "total_part1_violent",
            "total_part1_property"
        ])
    ]
    .groupby(["report_period", "district"], as_index=False)["current_month"]
    .sum()
)

missing_totals["section"] = "part1_total"
missing_totals["category"] = "total_part1"

overall_crime = pd.concat(
    [overall_crime, missing_totals],
    ignore_index=True
).sort_values(["report_period", "district"]).reset_index(drop=True)

print(overall_crime.shape)
print(overall_crime["report_period"].value_counts().sort_index().tail())

(440, 5)
report_period
2026-04    10
2026-05    10
2026-06    10
2026-07    10
2026-08    10
Name: count, dtype: int64


In [33]:
df["section"].unique()

<StringArray>
[    'part1_violent',    'part1_property',       'part1_total',
             'other',           'firearm', 'domestic_violence']
Length: 6, dtype: str

In [34]:
print(df.shape)
print(df["report_period"].min(), df["report_period"].max())

(10625, 13)
2023-01 2026-08


In [35]:
# Final crime handoff files

# 1. Category-level monthly crime data
crime_cleaned = crime_analysis[
    [
        "report_period",
        "district",
        "section",
        "category",
        "current_month"
    ]
].copy()

# 2. Overall Part I crime by district/month
crime_by_district_month = overall_crime[
    [
        "report_period",
        "district",
        "current_month"
    ]
].copy()

crime_by_district_month = crime_by_district_month.rename(
    columns={"current_month": "total_part1"}
)

# Save both files
crime_cleaned.to_csv(
    "crime_cleaned.csv",
    index=False
)

crime_by_district_month.to_csv(
    "crime_by_district_month.csv",
    index=False
)

# Validation
print("Category-level crime:", crime_cleaned.shape)
print("Overall district-month crime:", crime_by_district_month.shape)

print("\nDate range:")
print(
    crime_by_district_month["report_period"].min(),
    "to",
    crime_by_district_month["report_period"].max()
)

print("\nDistricts:")
print(
    sorted(crime_by_district_month["district"].unique())
)

print("\nDuplicate district-month rows:",
      crime_by_district_month[
          ["report_period", "district"]
      ].duplicated().sum())

Category-level crime: (10625, 5)
Overall district-month crime: (440, 3)

Date range:
2023-01 to 2026-08

Districts:
['BAYVIEW', 'CENTRAL', 'INGLESIDE', 'MISSION', 'NORTHERN', 'PARK', 'RICHMOND', 'SOUTHERN', 'TARAVAL', 'TENDERLOIN']

Duplicate district-month rows: 0
